In [1]:
import json
import numpy as np
from PIL import Image
import os


In [2]:
COLOR_TO_LABEL = {
    (0, 9, 255): "factory",
    (255, 0, 0): "brick house",
    (171, 171, 171): "billboard",
    (48, 135, 255): "rails",
    (102, 212, 0): "car",
    (255, 132, 0): "monolith house",
    (255, 213, 168): "bus",
    (13, 207, 255): "bus stop",
    (0, 150, 118): "road signs",
    (251, 255, 0): "traffic light",
    (255, 255, 255): "road",
    (255, 0, 204): "street light",
    (0, 255, 43): "panel house",
    (255, 184, 43): "fence",
    (119, 0, 179): "gravel",
    (138, 173, 255): "gas station",
    (189, 255, 194): "train",
    (71, 49, 29): "fountain",
    (148, 0, 89): "bench",
    (242, 82, 82): "tree",
    (66, 0, 0): "playground",
    (126, 98, 196): "wires",
    (255, 31, 236) : "other"

}

In [ ]:
 
# COLOR_TO_LABEL = {
#   (70, 70, 70): "building",
#   (0, 0, 142): "car",
#   (0, 60, 100): "bus",
#   (0, 80, 100): "train",
#   (230, 150, 140): "rails",
#   (128, 64, 128): "road",  
#   (153, 153, 153): "pole",  
#   (190, 153, 153): "fence",
#   (220, 220, 0): "traffic sign",  
#   (250, 170, 30): "traffic light",
#   (107, 142, 35): "tree",
#   (152, 251, 152): "gravel",  
#   (81, 0, 81): "ground", 
#   (0, 0, 0):  "unlabeled"


#  }

In [3]:
images_dir = r'C:\Users\KamarencevN\Desktop\диплом\Stable_diffusion\768.02\images'  # тут .jpeg
masks_dir = r'C:\Users\KamarencevN\Desktop\диплом\Stable_diffusion\768.02\masks'    # тут .png
output_file = "metadata.jsonl"

In [ ]:
def get_objects_from_mask(mask_path):
    mask = Image.open(mask_path).convert('RGB')
    mask_np = np.array(mask)
    
    objects_found = set()
    height, width, _ = mask_np.shape
    
    unique_colors = np.unique(mask_np.reshape(-1, 3), axis=0)
    
    for color in unique_colors:
        color_tuple = tuple(color)
        if color_tuple in COLOR_TO_LABEL:
            objects_found.add(COLOR_TO_LABEL[color_tuple])
    
    return sorted(list(objects_found))

with open(output_file, 'w', encoding='utf-8') as f:
    # Получаем все изображения
    for img_file in sorted(os.listdir(images_dir)):
        if img_file.endswith('.jpeg') or img_file.endswith('.jpg'):
            # Ищем соответствующую маску
            base_name = os.path.splitext(img_file)[0]
            mask_file = base_name + '.png'
            mask_path = os.path.join(masks_dir, mask_file)
            
            if os.path.exists(mask_path):
                # Получаем объекты из маски
                objects = get_objects_from_mask(mask_path)
                
                if objects:
                    text = "sks, " + ", ".join(objects)
                else:
                    text = "sks"
                
                entry = {
                    "file_name": img_file,
                    "text": text
                }
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
                print(f"Обработано: {img_file} -> {text}")
            else:
                print(f"Предупреждение: нет маски для {img_file}")

print(f"Готово! Файл {output_file} создан.")

Обработано: img_000000.jpeg -> sks, billboard, brick house, bus, bus stop, factory, fence, gravel, monolith house, panel house, road, traffic light, wires
Обработано: img_000001.jpeg -> sks, billboard, brick house, bus, bus stop, car, factory, fence, monolith house, panel house, road, traffic light, wires
Обработано: img_000002.jpeg -> sks, billboard, brick house, bus, bus stop, car, factory, fence, monolith house, panel house, road, traffic light, tree, wires
Обработано: img_000003.jpeg -> sks, billboard, brick house, bus, bus stop, car, factory, fence, monolith house, panel house, road, traffic light, tree, wires
Обработано: img_000004.jpeg -> sks, billboard, brick house, bus, bus stop, car, factory, fence, monolith house, panel house, road, traffic light, tree, wires
Обработано: img_000005.jpeg -> sks, billboard, brick house, bus, bus stop, car, factory, fence, monolith house, panel house, road, road signs, traffic light, tree, wires
Обработано: img_000006.jpeg -> sks, billboard, br

In [ ]:
import json
import os

input_file = r'C:\Users\KamarencevN\Desktop\диплом\Stable_diffusion\768\metadata_768.jsonl'
output_file = r'C:\Users\KamarencevN\Desktop\диплом\Stable_diffusion\768\metadata_fixed.jsonl'

with open(input_file, 'r', encoding='utf-8') as fin, \
     open(output_file, 'w', encoding='utf-8') as fout:
    
    for line in fin:
        data = json.loads(line.strip())
        img_name = data['file_name']
        cond_name = img_name.rsplit('.', 1)[0] + '.png'
        data['condition_file'] = cond_name
        fout.write(json.dumps(data, ensure_ascii=False) + '\n')

print(f"✅ Готово! Новый файл: {output_file}")

✅ Готово! Новый файл: C:\Users\KamarencevN\Desktop\диплом\Stable_diffusion\768\metadata_fixed.jsonl
